**Tema:** recursión

---

## Sección 7. Recursión

Una función recursiva es una función que **se llama a sí misma**. Siempre tiene dos partes:

- **Caso base:** la situación en la que la función ya no necesita llamarse y devuelve un resultado directamente. Sin él, la recursión no termina nunca.
- **Caso recursivo:** la función resuelve una parte del problema y se llama a sí misma con un problema **más pequeño**.

```python
def funcion(datos):
    if condicion_de_parada:              # caso base
        return resultado_directo
    return combinar(parte, funcion(resto))   # caso recursivo
```

La recursión es útil en dos tipos de problema:

1. **Problemas que se dividen en subproblemas iguales pero más pequeños.** Por ejemplo, el tamaño de una carpeta es la suma del tamaño de sus subcarpetas.
2. **Problemas con un número desconocido de bucles anidados.** Por ejemplo, recorrer un JSON del que no se sabe cuántos niveles tiene, o combinar los valores de un número variable de parámetros.

En ingeniería de datos y en IA aparece al recorrer **estructuras anidadas**: documentos JSON, árboles de carpetas de un data lake, grafos de dependencias, planes de un agente o trazas de ejecución.

### Ejemplos resueltos

#### Ejemplo 7.1 · Primera función recursiva con traza

In [0]:
def total_bytes(tamanos):
    if not tamanos:                                  # caso base: lista vacía
        return 0
    return tamanos[0] + total_bytes(tamanos[1:])     # caso recursivo

print(total_bytes([512, 1024, 256]))

# Misma función con traza para ver las llamadas
def total_bytes_traza(tamanos, nivel=0):
    sangria = "  " * nivel
    print(f"{sangria}total_bytes({tamanos})")
    if not tamanos:
        print(f"{sangria}-> 0  (caso base)")
        return 0
    resultado = tamanos[0] + total_bytes_traza(tamanos[1:], nivel + 1)
    print(f"{sangria}->{resultado}")
    return resultado

total_bytes_traza([512, 1024, 256])

1792
total_bytes([512, 1024, 256])
  total_bytes([1024, 256])
    total_bytes([256])
      total_bytes([])
      -> 0  (caso base)
    ->256
  ->1280
->1792


1792

**Paso a paso**

1. Caso base: una lista vacía suma 0 bytes.
2. Caso recursivo: el total es el primer elemento más el total del resto de la lista (`tamanos[1:]`), que es una lista más corta.
3. La traza muestra que las llamadas **bajan** hasta el caso base y que las sumas se resuelven al **subir**: 0 → 256 → 1280 → 1792. Mientras tanto, cada suma queda pendiente esperando el resultado de la llamada siguiente.
4. Cada llamada tiene sus propias variables. No se comparten entre llamadas.

#### Ejemplo 7.2 · Recursión de cola y recursión de cabeza

Según dónde esté la llamada recursiva, cambia el orden en que se hace el trabajo.

In [0]:
def procesar_cola(lotes):
    """Recursión de cola: primero procesa, después se llama."""
    if not lotes:
        return
    print("cola     ->", lotes[0])
    procesar_cola(lotes[1:])

def procesar_cabeza(lotes):
    """Recursión de cabeza: primero se llama, después procesa."""
    if not lotes:
        return
    procesar_cabeza(lotes[1:])
    print("cabeza   ->", lotes[0])

particiones = ["day=01", "day=02", "day=03"]
procesar_cola(particiones)
procesar_cabeza(particiones)

cola     -> day=01
cola     -> day=02
cola     -> day=03
cabeza   -> day=03
cabeza   -> day=02
cabeza   -> day=01



**Paso a paso**

1. **Recursión de cola:** la función procesa el primer elemento y **después** se llama con el resto. El trabajo se hace al bajar, en el orden original, como en un bucle.
2. **Recursión de cabeza:** la función se llama **primero** y procesa el elemento después, cuando la llamada interior ya ha terminado. El trabajo se hace al subir, así que el orden se invierte.
3. Las dos versiones tienen el mismo caso base y la misma llamada. Solo cambia el lugar del `print()`.

#### Ejemplo 7.3 · De un bucle a una función recursiva

Una política de reintentos con *backoff* exponencial duplica la espera tras cada fallo: 2 s, 4 s, 8 s… Queremos saber cuánto tiempo se espera en total.

In [0]:
# Versión iterativa: espera total de una política de reintentos
def espera_total_iterativa(reintentos, espera_inicial):
    total = 0
    espera = espera_inicial
    while reintentos > 0:
        total += espera
        espera *= 2
        reintentos -= 1
    return total

# Versión recursiva, siguiendo los pasos de conversión
def espera_total(reintentos, espera):
    if reintentos == 0:                                  # condición del bucle -> caso base
        return 0
    return espera + espera_total(reintentos - 1, espera * 2)   # cuerpo -> caso recursivo

print("Iterativa:", espera_total_iterativa(5, 2), "s")
print("Recursiva:", espera_total(5, 2), "s")
for n in range(1, 6):
    print(f"{n} reintentos ->{espera_total(n, 2):>2} s de espera acumulada")

Iterativa: 62 s
Recursiva: 62 s
1 reintentos -> 2 s de espera acumulada
2 reintentos -> 6 s de espera acumulada
3 reintentos ->14 s de espera acumulada
4 reintentos ->30 s de espera acumulada
5 reintentos ->62 s de espera acumulada


**Paso a paso**

Para convertir un bucle en recursión se siguen estos pasos:

1. Las variables que cambia el bucle (`reintentos`, `espera`) pasan a ser **parámetros** de la función recursiva.
2. La condición en la que el bucle termina (`reintentos == 0`) se convierte en el **caso base**.
3. El cuerpo del bucle se convierte en el **caso recursivo**: se suma la espera actual y se llama con un reintento menos y la espera duplicada.
4. Se comprueba que las dos versiones dan el mismo resultado y se eliminan las variables que ya no hacen falta. La versión recursiva no necesita el acumulador `total`.

#### Ejemplo 7.4 · Tamaño de un data lake con carpetas anidadas

In [0]:
# Carpetas = diccionarios; ficheros = tamaño en MB (int)
lake = {
    "bronce": {
        "ventas": {"2026-09.csv": 120, "2026-10.csv": 135},
        "clientes.csv": 40,
    },
    "plata": {
        "ventas": {"part-0000.parquet": 30, "part-0001.parquet": 28},
    },
    "oro": {"kpis_mensuales.parquet": 5},
}

def tamano_total(nodo):
    if isinstance(nodo, int):          # caso base: es un fichero
        return nodo
    total = 0
    for hijo in nodo.values():         # caso recursivo: es una carpeta
        total += tamano_total(hijo)
    return total

print("Data lake completo:", tamano_total(lake), "MB")
for capa, contenido in lake.items():
    print(f"{capa:<6}{tamano_total(contenido):>4} MB")

Data lake completo: 358 MB
bronce 295 MB
plata   58 MB
oro      5 MB


**Paso a paso**

1. `isinstance(nodo, int)` distingue un fichero, que es el caso base, de una carpeta, que es un diccionario.
2. Para una carpeta se suman los tamaños de todos sus hijos, y cada hijo puede ser a su vez otra carpeta.
3. Con un bucle `for` normal habría que saber de antemano cuántos niveles hay. La recursión funciona con cualquier profundidad.

#### Ejemplo 7.5 · Aplanar un JSON anidado

Antes de cargar un JSON en una tabla, hay que convertir sus objetos anidados en columnas planas, como hace la función `flatten` de muchas herramientas ETL.

In [0]:
evento = {
    "id": "e-1001",
    "cliente": {
        "id": "C001",
        "direccion": {"ciudad": "Madrid", "cp": "28001"},
    },
    "importe": 99.9,
    "dispositivo": {"tipo": "movil", "so": {"nombre": "Android", "version": "14"}},
}

def aplanar(diccionario, prefijo=""):
    plano = {}
    for clave, valor in diccionario.items():
        nombre = f"{prefijo}.{clave}" if prefijo else clave
        if isinstance(valor, dict):
            plano.update(aplanar(valor, nombre))    # caso recursivo
        else:
            plano[nombre] = valor                   # caso base
    return plano

for columna, valor in aplanar(evento).items():
    print(f"{columna:<26}{valor}")

id                        e-1001
cliente.id                C001
cliente.direccion.ciudad  Madrid
cliente.direccion.cp      28001
importe                   99.9
dispositivo.tipo          movil
dispositivo.so.nombre     Android
dispositivo.so.version    14


**Paso a paso**

1. `prefijo` acumula la ruta de claves: `cliente` → `cliente.direccion` → `cliente.direccion.ciudad`.
2. Si el valor es un diccionario, se aplana recursivamente y su resultado se incorpora con `update()`.
3. Si el valor es simple, es una hoja: se guarda con su nombre completo.

#### Ejemplo 7.6 · Orden de ejecución de un DAG de tareas

En un orquestador como Airflow, una tarea solo puede ejecutarse cuando han terminado todas sus dependencias.

In [0]:
dependencias = {
    "kpis_ventas": ["ventas_limpias", "clientes_limpios"],
    "ventas_limpias": ["ventas_raw"],
    "clientes_limpios": ["clientes_raw"],
    "ventas_raw": [],
    "clientes_raw": [],
}

def orden_ejecucion(tarea, ejecutadas):
    for previa in dependencias[tarea]:
        if previa not in ejecutadas:
            orden_ejecucion(previa, ejecutadas)     # primero, sus dependencias
    ejecutadas.append(tarea)                        # después, la propia tarea
    return ejecutadas

orden = orden_ejecucion("kpis_ventas", [])
for paso, tarea in enumerate(orden, start=1):
    print(paso, tarea)

1 ventas_raw
2 ventas_limpias
3 clientes_raw
4 clientes_limpios
5 kpis_ventas


**Paso a paso**

1. Antes de ejecutar una tarea, se ejecutan recursivamente sus dependencias.
2. La comprobación `if previa not in ejecutadas` evita repetir una tarea que comparte dependencias con otra.
3. La tarea se añade a la lista **después** de sus dependencias. Por eso `kpis_ventas` aparece en último lugar. Es una recursión de cabeza: el trabajo se hace al volver de las llamadas.

#### Ejemplo 7.7 · Búsqueda binaria y límite de recursión

Un fichero de log guarda un índice ordenado de *offsets*: la posición en bytes en la que empieza cada bloque. Buscar en él por mitades es mucho más rápido que recorrerlo entero.

In [0]:
import sys

def buscar(ordenada, objetivo, inicio, fin):
    if inicio > fin:                     # caso base 1: no está
        return -1
    medio = (inicio + fin) // 2
    if ordenada[medio] == objetivo:      # caso base 2: encontrado
        return medio
    if objetivo < ordenada[medio]:
        return buscar(ordenada, objetivo, inicio, medio - 1)
    return buscar(ordenada, objetivo, medio + 1, fin)

offsets = [0, 120, 245, 380, 512, 640, 777, 901, 1024]
print("640 está en la posición", buscar(offsets, 640, 0, len(offsets) - 1))
print("300 está en la posición", buscar(offsets, 300, 0, len(offsets) - 1))
print("Límite de recursión:", sys.getrecursionlimit())

def profundidad_lineal(n):
    if n == 0:
        return 0
    return 1 + profundidad_lineal(n - 1)

print(profundidad_lineal(900))
# profundidad_lineal(5000)  -> RecursionError: maximum recursion depth exceeded

sys.setrecursionlimit(10_000)      # se puede subir, con precaución
print(profundidad_lineal(5000))

640 está en la posición 5
300 está en la posición -1
Límite de recursión: 3000
900
5000


**Paso a paso**

1. En cada llamada se mira el elemento central. Si no es el buscado, se repite la búsqueda solo en la mitad donde puede estar.
2. Hay dos casos base: el elemento se encuentra, o el rango queda vacío (`inicio > fin`).
3. Con 1.000.000 de elementos bastan unas 20 llamadas, porque cada una descarta la mitad de los datos.
4. Python limita la profundidad de recursión a 1000 llamadas por defecto. `profundidad_lineal(900)` funciona, pero con 5000 se produce un `RecursionError`.
5. `sys.setrecursionlimit()` permite subir el límite, pero cada llamada pendiente ocupa memoria, y un límite demasiado alto puede hacer que el programa se cierre de golpe. Por eso la recursión es adecuada para estructuras con pocos niveles, como un JSON o un árbol de carpetas, y no para recorrer millones de elementos de uno en uno.

#### Ejemplo 7.8 · Generar todas las configuraciones de un benchmark de Spark

Antes de llevar un job a producción, se prueba con distintas configuraciones de recursos. Con tres parámetros harían falta tres bucles anidados, pero si mañana se añade un cuarto parámetro, habría que reescribir el código. Es un problema con un **número desconocido de bucles**.

In [0]:
parametros = {
    "executors": [4, 8],
    "memoria": ["4g", "8g"],
    "shuffle_partitions": [200, 400],
}

def combinaciones(nombres, parametros, actual, resultado):
    if len(actual) == len(nombres):          # caso base: ya hay un valor por parámetro
        resultado.append(dict(zip(nombres, actual)))
        return
    nombre = nombres[len(actual)]            # siguiente parámetro a fijar
    for valor in parametros[nombre]:
        actual.append(valor)
        combinaciones(nombres, parametros, actual, resultado)
        actual.pop()                         # deshacer para probar el siguiente valor

configuraciones = []
combinaciones(list(parametros), parametros, [], configuraciones)
for i, conf in enumerate(configuraciones, start=1):
    print(i, conf)

# Añadir un parámetro más no exige añadir un bucle más
parametros["compresion"] = ["snappy", "zstd"]
configuraciones = []
combinaciones(list(parametros), parametros, [], configuraciones)
print("Con 4 parámetros:", len(configuraciones), "configuraciones")

1 {'executors': 4, 'memoria': '4g', 'shuffle_partitions': 200}
2 {'executors': 4, 'memoria': '4g', 'shuffle_partitions': 400}
3 {'executors': 4, 'memoria': '8g', 'shuffle_partitions': 200}
4 {'executors': 4, 'memoria': '8g', 'shuffle_partitions': 400}
5 {'executors': 8, 'memoria': '4g', 'shuffle_partitions': 200}
6 {'executors': 8, 'memoria': '4g', 'shuffle_partitions': 400}
7 {'executors': 8, 'memoria': '8g', 'shuffle_partitions': 200}
8 {'executors': 8, 'memoria': '8g', 'shuffle_partitions': 400}
Con 4 parámetros: 16 configuraciones


**Paso a paso**

1. `actual` es la configuración que se está construyendo. Cada nivel de recursión fija el valor de un parámetro.
2. **Caso base:** cuando `actual` tiene un valor por cada parámetro, la configuración está completa y se guarda como diccionario con `dict(zip(...))`.
3. **Caso recursivo:** para cada valor del siguiente parámetro, se añade a `actual`, se llama a la función para fijar el resto y, al volver, se quita con `pop()` para probar el siguiente valor. Esta técnica se llama *backtracking*.
4. Al añadir `compresion`, el código no cambia y genera 2 × 2 × 2 × 2 = 16 configuraciones.

#### Ejemplo 7.9 · Fusionar particiones ordenadas (divide y vencerás)

Cuando Spark ordena un dataset grande, cada partición se ordena por separado y después se combinan los resultados. La función recursiva divide la lista de particiones por la mitad hasta quedarse con una sola, y al volver fusiona los resultados de dos en dos.

In [0]:
def fusionar(a, b):
    """Une dos listas ordenadas en una sola lista ordenada."""
    resultado = []
    i = j = 0
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            resultado.append(a[i])
            i += 1
        else:
            resultado.append(b[j])
            j += 1
    return resultado + a[i:] + b[j:]

def fusionar_particiones(particiones, nivel=0):
    print("  " * nivel + f"fusionar{len(particiones)} particiones")
    if len(particiones) == 1:                       # caso base
        return particiones[0]
    mitad = len(particiones) // 2
    izquierda = fusionar_particiones(particiones[:mitad], nivel + 1)
    derecha = fusionar_particiones(particiones[mitad:], nivel + 1)
    return fusionar(izquierda, derecha)

# Timestamps ya ordenados dentro de cada partición
particiones = [[3, 9, 15], [1, 12], [5, 6, 20], [2, 18]]
ordenado = fusionar_particiones(particiones)
print("Resultado:", ordenado)

fusionar4 particiones
  fusionar2 particiones
    fusionar1 particiones
    fusionar1 particiones
  fusionar2 particiones
    fusionar1 particiones
    fusionar1 particiones
Resultado: [1, 2, 3, 5, 6, 9, 12, 15, 18, 20]


**Paso a paso**

1. `fusionar()` no es recursiva: une dos listas ya ordenadas comparando sus primeros elementos y tomando siempre el menor.
2. `fusionar_particiones()` divide la lista de particiones en dos mitades y se llama con cada una.
3. **Caso base:** una sola partición ya está ordenada y se devuelve tal cual.
4. Al volver de las dos llamadas, se fusionan los dos resultados. La traza muestra el árbol de llamadas: 4 → 2 + 2 → 1 + 1 + 1 + 1.
5. Este mismo esquema, dividir el problema, resolver cada parte y combinar los resultados, se llama **divide y vencerás**, y es la base del algoritmo *merge sort*.

#### Ejemplo 7.10 · El plan jerárquico de un agente de IA

Un agente de IA que recibe una tarea compleja la **descompone** en subtareas, y cada subtarea puede descomponerse a su vez, hasta llegar a acciones concretas que se resuelven con una herramienta (una consulta SQL, una búsqueda web, una llamada al LLM…). El plan resultante es un árbol.

In [0]:
plan = {
    "tarea": "Informe de ventas de octubre",
    "subtareas": [
        {"tarea": "Obtener datos", "subtareas": [
            {"tarea": "Consultar ventas", "herramienta": "sql"},
            {"tarea": "Consultar objetivos", "herramienta": "api_crm"},
        ]},
        {"tarea": "Analizar", "subtareas": [
            {"tarea": "Calcular KPIs", "herramienta": "python"},
            {"tarea": "Detectar anomalías", "subtareas": [
                {"tarea": "Comparar con septiembre", "herramienta": "python"},
                {"tarea": "Buscar causas en noticias", "herramienta": "web_search"},
            ]},
        ]},
        {"tarea": "Redactar el informe", "herramienta": "llm"},
    ],
}

def mostrar_plan(nodo, nivel=0):
    if "herramienta" in nodo:                         # caso base: acción concreta
        print("  " * nivel + f"-{nodo['tarea']}  [{nodo['herramienta']}]")
        return 1
    print("  " * nivel + f"+{nodo['tarea']}")
    acciones = 0
    for sub in nodo["subtareas"]:                     # caso recursivo: descomponer
        acciones += mostrar_plan(sub, nivel + 1)
    return acciones

total = mostrar_plan(plan)
print("Acciones que ejecutará el agente:", total)

+Informe de ventas de octubre
  +Obtener datos
    -Consultar ventas  [sql]
    -Consultar objetivos  [api_crm]
  +Analizar
    -Calcular KPIs  [python]
    +Detectar anomalías
      -Comparar con septiembre  [python]
      -Buscar causas en noticias  [web_search]
  -Redactar el informe  [llm]
Acciones que ejecutará el agente: 6


**Paso a paso**

1. Cada nodo del plan es un diccionario. Si tiene la clave `herramienta`, es una **acción concreta**: el caso base. Se imprime y cuenta como 1.
2. Si tiene `subtareas`, es una tarea compuesta: se imprime y se llama a la función con cada subtarea, un nivel más adentro.
3. `nivel` controla la sangría, así que la salida refleja la estructura del árbol.
4. La función devuelve el número de acciones de cada rama, y la tarea compuesta suma las de sus hijos. Así sabemos cuántas herramientas tendrá que ejecutar el agente.

#### Ejemplo 7.11 · *Chunking* recursivo para un sistema RAG

Los *splitters* recursivos de herramientas como LangChain intentan cortar un documento por los separadores más “naturales” primero: párrafos, después líneas, después frases y, solo si no queda más remedio, palabras. Así cada chunk respeta al máximo la estructura del texto.

In [0]:
def trocear(texto, max_caracteres, separadores=("\n\n", "\n", ". ", " ")):
    if len(texto) <= max_caracteres:                  # caso base: ya cabe
        return [texto]
    if not separadores:                               # caso base: corte duro
        return [texto[i:i + max_caracteres] for i in range(0, len(texto), max_caracteres)]
    separador = separadores[0]
    piezas = []
    for parte in texto.split(separador):              # caso recursivo
        if parte.strip():
            piezas.extend(trocear(parte.strip(), max_caracteres, separadores[1:]))
    chunks = []                                       # juntar piezas pequeñas
    for pieza in piezas:
        if chunks and len(chunks[-1]) + len(separador) + len(pieza) <= max_caracteres:
            chunks[-1] = chunks[-1] + separador + pieza
        else:
            chunks.append(pieza)
    return chunks

manual = """Política de envíos. Los pedidos se envían en 24 horas.

Devoluciones. El plazo es de 30 días. El producto debe estar sin usar. Los gastos de devolución corren a cargo del cliente salvo defecto de fábrica.

Garantía. Dos años."""

for i, chunk in enumerate(trocear(manual, 60)):
    print(f"{i} ({len(chunk):>2} car.){chunk!r}")

0 (54 car.)'Política de envíos. Los pedidos se envían en 24 horas.'
1 (36 car.)'Devoluciones. El plazo es de 30 días'
2 (31 car.)'El producto debe estar sin usar'
3 (57 car.)'Los gastos de devolución corren a cargo del cliente salvo'
4 (40 car.)'defecto de fábrica.\n\nGarantía. Dos años.'


**Paso a paso**

1. **Caso base 1:** si el texto ya cabe en el límite, se devuelve tal cual.
2. **Caso base 2:** si no quedan separadores, se corta a la fuerza cada `max_caracteres`.
3. **Caso recursivo:** se divide el texto con el primer separador y cada parte se trocea de nuevo con los separadores **restantes**. Una frase demasiado larga acabará dividida por espacios.
4. Al volver, se juntan las piezas pequeñas consecutivas mientras quepan en el límite. Sin este paso, la frase larga quedaría convertida en palabras sueltas.
5. Se muestra cada chunk con `!r` para que se vea el salto de párrafo (`\n\n`) del último.

### Ejercicios propuestos

#### Ejercicio 7.1 · Profundidad máxima de un documento JSON

Algunos sistemas de destino limitan el número de niveles de anidamiento que aceptan. Escribe `profundidad(nodo)` con estas reglas: un valor simple tiene profundidad 0, y un diccionario o una lista tienen profundidad 1 más la mayor profundidad de sus elementos. Un diccionario o una lista vacíos tienen profundidad 1.
Calcula la profundidad de `documento`, de `documento["items"]` y del valor `42`.

In [0]:
documento = {
    "id": 1,
    "cliente": {"nombre": "Acme", "contacto": {"email": "a@acme.com", "tel": {"pref": "+34"}}},
    "items": [{"sku": "A1", "atributos": {"color": "rojo"}}, {"sku": "B2"}],
}

def profundidad(nodo):
    if isinstance(nodo, dict):
        if not nodo:
            return 1
        return 1 + max(profundidad(v) for v in nodo.values())
    elif isinstance(nodo, list):
        if not nodo:
            return 1
        return 1 + max(profundidad(elem) for elem in nodo)
    else:
        return 0

print("Profundidad:", profundidad(documento))
print("Profundidad de 'items':", profundidad(documento["items"]))
print("Profundidad de un valor simple:", profundidad(42))

Profundidad: 4
Profundidad de 'items': 3
Profundidad de un valor simple: 0


#### Ejercicio 7.2 · Localizar todos los ficheros Parquet
Escribe `buscar_ficheros(nodo, extension, ruta="")`, que devuelva la lista de rutas completas de los ficheros con esa extensión. Usa `nombre.endswith(extension)`. Muestra las rutas `.parquet` y su total, y después solo el total de `.csv`.


In [0]:
lake = {
    "bronce": {"ventas": {"2026-09.csv": 120, "2026-10.csv": 135}, "clientes.csv": 40},
    "plata": {
        "ventas": {"year=2026": {"part-0000.parquet": 30, "part-0001.parquet": 28}},
        "clientes": {"part-0000.parquet": 12},
    },
    "oro": {"kpis_mensuales.parquet": 5, "README.md": 1},
}

def buscar_ficheros(nodo, extension, ruta=""):
    rutas = []
    if isinstance(nodo, dict):
        for nombre, subnodo in nodo.items():
            rutas.extend(buscar_ficheros(subnodo, extension, f"{ruta}/{nombre}"))
    else:
        if ruta.endswith(extension):
            rutas.append(ruta)
    return rutas

ficheros_parquet = buscar_ficheros(lake, ".parquet")
for f in ficheros_parquet:
    print(f)
print("Total .parquet:", len(ficheros_parquet))

ficheros_csv = buscar_ficheros(lake, ".csv")
print("Total .csv:", len(ficheros_csv))

/plata/ventas/year=2026/part-0000.parquet
/plata/ventas/year=2026/part-0001.parquet
/plata/clientes/part-0000.parquet
/oro/kpis_mensuales.parquet
Total .parquet: 4
Total .csv: 3


#### Ejercicio 7.3 · Linaje de datos: ¿de qué tablas depende un dashboard?

Este diccionario indica las tablas de origen directas de cada tabla.
Escribe `ancestros(tabla)`, que devuelva un `set` con todas las tablas de las que depende, directa o indirectamente. Muestra los ancestros de `dashboard_ventas`, de `ventas_limpias` y de `ventas_raw`. Por último, con una comprehension, obtén las fuentes originales del dashboard: los ancestros que no tienen ningún origen.

In [0]:
origenes = {
    "dashboard_ventas": ["kpis_ventas", "dim_tiempo"],
    "kpis_ventas": ["ventas_limpias", "clientes_limpios"],
    "ventas_limpias": ["ventas_raw", "tipos_cambio"],
    "clientes_limpios": ["clientes_raw"],
    "dim_tiempo": [],
    "ventas_raw": [],
    "clientes_raw": [],
    "tipos_cambio": [],
}

def ancestros(tabla):
    resultado = set()
    for origen in origenes.get(tabla, []):
        resultado.add(origen)
        resultado.update(ancestros(origen))
    return resultado

print(sorted(ancestros("dashboard_ventas")))
print(sorted(ancestros("ventas_limpias")))
print(sorted(ancestros("ventas_raw")))

dashboard_ancestros = ancestros("dashboard_ventas")
fuentes_originales = sorted([t for t in dashboard_ancestros if len(origenes[t]) == 0])
print("Fuentes originales del dashboard:", fuentes_originales)

['clientes_limpios', 'clientes_raw', 'dim_tiempo', 'kpis_ventas', 'tipos_cambio', 'ventas_limpias', 'ventas_raw']
['tipos_cambio', 'ventas_raw']
[]
Fuentes originales del dashboard: ['clientes_raw', 'dim_tiempo', 'tipos_cambio', 'ventas_raw']


#### Ejercicio 7.4 · Contar nulos en una respuesta de API

La respuesta mezcla diccionarios y listas a varios niveles.
Escribe `contar_nulos(nodo)`, que cuente todos los `None` de la estructura, estén dentro de diccionarios o de listas.

In [0]:
respuesta_api = {
    "pedido": 9001,
    "cliente": {"nombre": "Acme", "email": None, "telefonos": ["600111222", None]},
    "lineas": [
        {"sku": "A1", "precio": 10.0, "descuento": None},
        {"sku": None, "precio": None, "descuento": 0.1},
    ],
    "notas": None,
}

def contar_nulos(nodo):
    if nodo is None:
        return 1
    elif isinstance(nodo, dict):
        return sum(contar_nulos(v) for v in nodo.values())
    elif isinstance(nodo, list):
        return sum(contar_nulos(elem) for elem in nodo)
    else:
        return 0

print("Nulos en la respuesta:", contar_nulos(respuesta_api))
print("Nulos en 'lineas':", contar_nulos(respuesta_api["lineas"]))

Nulos en la respuesta: 6
Nulos en 'lineas': 3


#### Ejercicio 7.5 · Detectar ciclos en las dependencias

Un orquestador rechaza un flujo de tareas si sus dependencias forman un ciclo (por ejemplo, `b → c → d → b`), porque ninguna de esas tareas podría arrancar nunca.
1. Escribe `tiene_ciclo(grafo, nodo, camino)`, que devuelva `True` si, partiendo de `nodo`, se vuelve a un nodo que ya está en el camino actual. `camino` es un `set` con los nodos visitados en la rama que se está recorriendo.
2. Escribe `validar_dag(grafo)`, que compruebe todos los nodos y devuelva `True` si no hay ningún ciclo.

In [0]:
dag_ok = {"a": ["b", "c"], "b": ["d"], "c": ["d"], "d": []}
dag_ko = {"a": ["b"], "b": ["c"], "c": ["d"], "d": ["b"]}

def tiene_ciclo(grafo, nodo, camino):
    if nodo in camino:
        return True
    camino.add(nodo)
    for vecino in grafo.get(nodo, []):
        if tiene_ciclo(grafo, vecino, camino):
            return True
    camino.remove(nodo)
    return False

def validar_dag(grafo):
    for nodo in grafo:
        if tiene_ciclo(grafo, nodo, set()):
            return False
    return True

print("dag_ok es un DAG válido:", validar_dag(dag_ok))
print("dag_ko es un DAG válido:", validar_dag(dag_ko))

dag_ok es un DAG válido: True
dag_ko es un DAG válido: False


#### Ejercicio 7.6 · Tamaños legibles

Los sistemas de almacenamiento informan del tamaño en bytes, pero en los informes se muestran en la unidad más adecuada: KB, MB, GB…
Escribe `legible(valor, nivel=0)` de forma **recursiva**, usando la lista de unidades `["B", "KB", "MB", "GB", "TB", "PB"]`:

- **Caso base:** si el valor es menor que 1024, o ya no quedan unidades mayores, se devuelve con un decimal y su unidad.
- **Caso recursivo:** se divide el valor entre 1024 y se llama con el nivel siguiente.

In [0]:
tamanos_bytes = [512, 2048, 5_368_709_120, 1_099_511_627_776, 734_003_200]
unidades = ["B", "KB", "MB", "GB", "TB", "PB"]

def legible(valor, nivel=0):
    if valor < 1024 or nivel == len(unidades) - 1:
        if nivel == 0:
            return f"{int(valor)} {unidades[nivel]}"
        return f"{valor:.1f} {unidades[nivel]}"
    return legible(valor / 1024, nivel + 1)

for tb in tamanos_bytes:
    print(f"{tb:>16} -> {legible(tb)}")

             512 -> 512 B
            2048 -> 2.0 KB
      5368709120 -> 5.0 GB
   1099511627776 -> 1.0 TB
       734003200 -> 700.0 MB


#### Ejercicio 7.7 · *Tree reduce* de conteos parciales

Cuando hay muchas particiones, combinar todos sus resultados en un solo nodo puede saturarlo. Spark ofrece `treeReduce()`, que combina los resultados parciales por parejas, en varios niveles, como un torneo.
1. Escribe `combinar(a, b)`, que devuelva un diccionario nuevo con la suma de los conteos de los dos diccionarios. No es recursiva.
2. Escribe `tree_reduce(parciales, nivel=0)`, que divida la lista en dos mitades, se llame con cada una y combine los dos resultados. El caso base es una lista con un solo diccionario. Imprime en cada llamada el nivel y el número de parciales que recibe, con una sangría de dos espacios por nivel.
3. Muestra el resultado final con las claves ordenadas.

In [0]:
parciales = [
    {"click": 120, "view": 400},
    {"click": 98, "compra": 7},
    {"view": 310, "compra": 3},
    {"click": 150, "view": 280, "compra": 12},
    {"view": 95},
    {"click": 40, "compra": 1},
]

def combinar(a, b):
    res = a.copy()
    for k, v in b.items():
        res[k] = res.get(k, 0) + v
    return res

def tree_reduce(parciales, nivel=0):
    indent = "  " * nivel
    print(f"{indent}nivel {nivel}: {len(parciales)} parciales")
    if len(parciales) == 1:
        return parciales[0]
    mitad = len(parciales) // 2
    izq = tree_reduce(parciales[:mitad], nivel + 1)
    der = tree_reduce(parciales[mitad:], nivel + 1)
    return combinar(izq, der)

resultado = tree_reduce(parciales)
resultado_ordenado = {k: resultado[k] for k in sorted(resultado)}
print("Resultado:", resultado_ordenado)

nivel 0: 6 parciales
  nivel 1: 3 parciales
    nivel 2: 1 parciales
    nivel 2: 2 parciales
      nivel 3: 1 parciales
      nivel 3: 1 parciales
  nivel 1: 3 parciales
    nivel 2: 1 parciales
    nivel 2: 2 parciales
      nivel 3: 1 parciales
      nivel 3: 1 parciales
Resultado: {'click': 408, 'compra': 23, 'view': 1085}


#### Ejercicio 7.8 · Detectar el problema de los ficheros pequeños

En HDFS y en los data lakes, miles de ficheros diminutos degradan el rendimiento: cada fichero supone metadatos y una tarea de lectura. Se recomienda compactar las carpetas donde predominan los ficheros pequeños.
1. Escribe `ficheros_pequenos(nodo, umbral)`, que recorra una carpeta y todas sus subcarpetas y devuelva una tupla `(total_de_ficheros, ficheros_por_debajo_del_umbral)`.
2. Aplícala a cada partición de fecha y marca `COMPACTAR` las que tengan más de la mitad de sus ficheros por debajo del umbral.
3. Aplícala también a la tabla completa.


In [0]:
tabla_eventos = {
    "fecha=2026-10-13": {"part-0.parquet": 210, "part-1.parquet": 190},
    "fecha=2026-10-14": {f"part-{i}.parquet": 3 for i in range(6)},
    "fecha=2026-10-15": {
        "hora=00": {"part-0.parquet": 2, "part-1.parquet": 4},
        "hora=01": {"part-0.parquet": 150},
    },
}
umbral_mb = 32

def ficheros_pequenos(nodo, umbral):
    total = 0
    pequenos = 0
    if isinstance(nodo, dict):
        for subnodo in nodo.values():
            t, p = ficheros_pequenos(subnodo, umbral)
            total += t
            pequenos += p
    else:
        total = 1
        if nodo < umbral:
            pequenos = 1
    return total, pequenos

for particion, datos in tabla_eventos.items():
    tot, peqs = ficheros_pequenos(datos, umbral_mb)
    estado = "COMPACTAR" if peqs > tot / 2 else "ok"
    print(f"{particion}: {tot} ficheros, {peqs} pequeños -> {estado}")

print("Tabla completa (ficheros, pequeños):", ficheros_pequenos(tabla_eventos, umbral_mb))

fecha=2026-10-13: 2 ficheros, 0 pequeños -> ok
fecha=2026-10-14: 6 ficheros, 6 pequeños -> COMPACTAR
fecha=2026-10-15: 3 ficheros, 2 pequeños -> COMPACTAR
Tabla completa (ficheros, pequeños): (11, 8)


#### Ejercicio 7.9 · Analizar la traza de ejecución de un agente

Las plataformas de observabilidad de agentes guardan cada ejecución como un árbol de pasos: llamadas al LLM, llamadas a herramientas y subagentes, que a su vez lanzan otros pasos.
Escribe cuatro funciones recursivas:

- `tokens_totales(nodo)`: suma los tokens del nodo y de todos sus descendientes.
- `profundidad(nodo)`: un paso sin hijos tiene profundidad 1.
- `herramientas(nodo)`: devuelve el `set` de herramientas usadas en los pasos de tipo `tool`.
- `llamadas_llm(nodo)`: cuenta los pasos de tipo `llm`.

Muestra los resultados para la traza completa y los tokens consumidos solo por `subagente_analista`.


In [0]:
traza = {
    "paso": "agente_principal", "tipo": "llm", "tokens": 1200,
    "hijos": [
        {"paso": "buscar_clientes", "tipo": "tool", "herramienta": "sql", "tokens": 0, "hijos": []},
        {"paso": "subagente_analista", "tipo": "llm", "tokens": 2300, "hijos": [
            {"paso": "calcular_churn", "tipo": "tool", "herramienta": "python", "tokens": 0, "hijos": []},
            {"paso": "buscar_contexto", "tipo": "tool", "herramienta": "rag", "tokens": 450, "hijos": [
                {"paso": "reescribir_consulta", "tipo": "llm", "tokens": 300, "hijos": []},
            ]},
        ]},
        {"paso": "redactar_respuesta", "tipo": "llm", "tokens": 1800, "hijos": []},
    ],
}

def tokens_totales(nodo):
    return nodo.get("tokens", 0) + sum(tokens_totales(h) for h in nodo.get("hijos", []))

def profundidad(nodo):
    hijos = nodo.get("hijos", [])
    if not hijos:
        return 1
    return 1 + max(profundidad(h) for h in hijos)

def herramientas(nodo):
    usadas = set()
    if nodo.get("tipo") == "tool" and "herramienta" in nodo:
        usadas.add(nodo["herramienta"])
    for h in nodo.get("hijos", []):
        usadas.update(herramientas(h))
    return usadas

def llamadas_llm(nodo):
    cuenta = 1 if nodo.get("tipo") == "llm" else 0
    return cuenta + sum(llamadas_llm(h) for h in nodo.get("hijos", []))

print("Tokens totales:", tokens_totales(traza))
print("Profundidad máxima:", profundidad(traza))
print("Herramientas usadas:", sorted(herramientas(traza)))
print("Llamadas al LLM:", llamadas_llm(traza))

subagente = next(h for h in traza["hijos"] if h["paso"] == "subagente_analista")
print("Tokens del subagente:", tokens_totales(subagente))

Tokens totales: 6050
Profundidad máxima: 4
Herramientas usadas: ['python', 'rag', 'sql']
Llamadas al LLM: 4
Tokens del subagente: 3050


#### Ejercicio 7.10 · Resumen jerárquico de documentos largos

Cuando un documento no cabe en el contexto de un LLM, se usa un resumen jerárquico: se resumen los fragmentos por grupos, después se resumen esos resúmenes, y así hasta obtener uno solo. Como aquí no llamamos a un modelo real, la función `resumir()` lo simula quedándose con las 3 primeras palabras de cada texto.


In [0]:
def resumir(textos):
    return " ".join([" ".join(t.split()[:3]) for t in textos])

capitulos = [
    "Ventas crecen un 12% en el canal online",
    "Tiendas físicas estables respecto al trimestre anterior",
    "Devoluciones bajan tras el cambio de política",
    "Coste logístico sube por el precio del combustible",
    "Nuevo almacén en Valencia reduce plazos de entrega",
]

def resumen_jerarquico(textos, grupo=2, nivel=0):
    print(f"Nivel {nivel}: {len(textos)} textos")
    if len(textos) <= 1:
        return textos[0] if textos else ""
    
    resumenes = [
        resumir(textos[i:i + grupo])
        for i in range(0, len(textos), grupo)
    ]
    return resumen_jerarquico(resumenes, grupo=grupo, nivel=nivel + 1)

res_final_2 = resumen_jerarquico(capitulos, grupo=2)
print("Resumen final:", res_final_2)
print()
res_final_3 = resumen_jerarquico(capitulos, grupo=3)
print("Resumen final:", res_final_3)

Nivel 0: 5 textos
Nivel 1: 3 textos
Nivel 2: 2 textos
Nivel 3: 1 textos
Resumen final: Ventas crecen un Nuevo almacén en

Nivel 0: 5 textos
Nivel 1: 2 textos
Nivel 2: 1 textos
Resumen final: Ventas crecen un Coste logístico sube
